In [1]:
import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import xml.etree.ElementTree as ET
from tqdm.notebook import tqdm

# --- UPDATED CONFIGURATION ---
CONFIG = {
    'xml_root': r'H:\DPJI\IDDPedestrian\annotations\gopro',
    'img_root': r'H:\DPJI\fast_data_static_v2', 
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'obs_len': 15,  
    'pred_len': 45, 
    'hidden_size': 256,
    'embed_size': 128, 
    'batch_size': 64,  
    'epochs': 30,      
    'lr': 5e-4
}

print(f"✅ Libraries Imported. Device: {CONFIG['device']}")

✅ Libraries Imported. Device: cuda


In [2]:
class IDDTrajectoryDataset(Dataset):
    def __init__(self, xml_root, obs_len=15, pred_len=45):
        self.obs_len, self.pred_len = obs_len, pred_len
        self.seq_len = obs_len + pred_len
        self.samples = []
        
        # Lists to collect all displacements for standardization
        all_disps = []

        xml_files = glob.glob(os.path.join(xml_root, '**', '*.xml'), recursive=True)
        for xml in tqdm(xml_files, desc="Parsing XMLs"):
            try:
                root = ET.parse(xml).getroot()
                for track in root.findall('track'):
                    if track.attrib['label'] != 'pedestrian': continue
                    
                    coords = []
                    for box in track.findall('box'):
                        coords.append([(float(box.attrib['xtl'])+float(box.attrib['xbr']))/2, 
                                       (float(box.attrib['ytl'])+float(box.attrib['ybr']))/2,
                                       float(box.attrib['xbr'])-float(box.attrib['xtl']), 
                                       float(box.attrib['ybr'])-float(box.attrib['ytl'])])
                    
                    coords = np.array(coords)
                    if len(coords) < self.seq_len: continue

                    for i in range(0, len(coords) - self.seq_len + 1, 5):
                        full_seq = coords[i : i + self.seq_len]
                        disp = np.diff(full_seq[:, :2], axis=0, prepend=full_seq[:1, :2])
                        all_disps.append(disp)
                        
                        self.samples.append({
                            'obs_disp': disp[:obs_len],
                            'last_pos': full_seq[obs_len-1, :2],
                            'abs_gt': full_seq[obs_len:, :2],
                            'wh_gt': full_seq[obs_len:, 2:]
                        })
            except: pass

        # Calculate Statistics for Standardization
        all_disps = np.concatenate(all_disps)
        self.mean = np.mean(all_disps, axis=0)
        self.std = np.std(all_disps, axis=0) + 1e-6
        print(f"📊 Stats - Mean: {self.mean}, Std: {self.std}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        # Standardize inputs
        norm_disp = (s['obs_disp'] - self.mean) / self.std
        return (
            torch.tensor(norm_disp, dtype=torch.float32),
            torch.tensor(s['last_pos'], dtype=torch.float32),
            torch.tensor(s['abs_gt'], dtype=torch.float32),
            torch.tensor(s['wh_gt'], dtype=torch.float32)
        )

dataset = IDDTrajectoryDataset(CONFIG['xml_root'])

Parsing XMLs:   0%|          | 0/33 [00:00<?, ?it/s]

📊 Stats - Mean: [-1.49183408  0.12157585], Std: [6.0783083 1.2980012]


In [3]:
class PIETrajNet(nn.Module):
    def __init__(self, input_size=2, hidden_size=256, embed_size=128):
        super(PIETrajNet, self).__init__()
        self.embed = nn.Sequential(nn.Linear(input_size, embed_size), nn.ReLU())
        self.encoder = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.decoder = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 2)
        
    def forward(self, obs_disp, pred_len):
        batch_size = obs_disp.size(0)
        emb = self.embed(obs_disp)
        _, (h, c) = self.encoder(emb)
        
        # Start decoding with the last observed displacement
        curr_disp = obs_disp[:, -1, :].unsqueeze(1)
        predictions = []
        
        for _ in range(pred_len):
            curr_emb = self.embed(curr_disp)
            out, (h, c) = self.decoder(curr_emb, (h, c))
            disp_pred = self.fc(out)
            predictions.append(disp_pred)
            curr_disp = disp_pred # Autoregressive loop
            
        return torch.cat(predictions, dim=1)

model = PIETrajNet(hidden_size=CONFIG['hidden_size'], embed_size=CONFIG['embed_size']).to(CONFIG['device'])

In [5]:
from torch.utils.data import DataLoader, random_split
import torch.nn as nn

# 1. Setup Data Loaders
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_set, val_set = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(val_set, batch_size=CONFIG['batch_size'])

# 2. Setup Optimizer and Scheduler
optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'])
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
criterion = nn.MSELoss()

# 3. Get Stats from Dataset for Un-standardization
# Important: Ensure you have run the updated Cell 2 before this!
D_MEAN = torch.tensor(dataset.mean, dtype=torch.float32).to(CONFIG['device'])
D_STD = torch.tensor(dataset.std, dtype=torch.float32).to(CONFIG['device'])

print(f"🚀 Training with Standardization...")

for epoch in range(CONFIG['epochs']):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False)
    
    for norm_disp, last_pos, abs_gt, _ in pbar:
        # Move everything to GPU/CPU
        norm_disp = norm_disp.to(CONFIG['device'])
        last_pos = last_pos.to(CONFIG['device'])
        abs_gt = abs_gt.to(CONFIG['device'])
        
        optimizer.zero_grad()
        
        # Predict & Un-standardize
        pred_norm_disps = model(norm_disp, CONFIG['pred_len'])
        pred_disps = pred_norm_disps * D_STD + D_MEAN
        
        # Reconstruct path [Batch, 45, 2]
        pred_abs = last_pos.unsqueeze(1) + torch.cumsum(pred_disps, dim=1)
        
        # Calculate Weighted Loss
        # main_loss = error over whole path
        # final_loss = error at 1.5s destination
        main_loss = criterion(pred_abs, abs_gt)
        final_loss = criterion(pred_abs[:, -1, :], abs_gt[:, -1, :])
        
        loss = main_loss + 0.5 * final_loss
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.2f}"})
        
    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{CONFIG['epochs']} | Loss: {avg_loss:.2f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

print("✅ Training Complete.")

🚀 Training with Standardization...


Epoch 1:   0%|          | 0/661 [00:00<?, ?it/s]

Epoch 1/30 | Loss: 7434.05 | LR: 0.000500


Epoch 2:   0%|          | 0/661 [00:00<?, ?it/s]

Epoch 2/30 | Loss: 6430.44 | LR: 0.000500


...

Epoch 28/30 | Loss: 4427.29 | LR: 0.000125


Epoch 29:   0%|          | 0/661 [00:00<?, ?it/s]

Epoch 29/30 | Loss: 4362.40 | LR: 0.000125


Epoch 30:   0%|          | 0/661 [00:00<?, ?it/s]

Epoch 30/30 | Loss: 4326.46 | LR: 0.000063
✅ Training Complete.


In [6]:
def evaluate_pietraj_v2(model, loader):
    model.eval()
    all_mse, all_cmse, all_cfmse = [], [], []
    
    with torch.no_grad():
        for norm_disp, last_pos, abs_gt, wh_gt in loader:
            norm_disp = norm_disp.to(CONFIG['device'])
            
            # Predict and Un-standardize
            pred_norm = model(norm_disp, CONFIG['pred_len'])
            pred_disps = (pred_norm * D_STD + D_MEAN).cpu().numpy()
            
            abs_gt, wh_gt, last_pos = abs_gt.numpy(), wh_gt.numpy(), last_pos.numpy()
            pred_abs = last_pos[:, np.newaxis, :] + np.cumsum(pred_disps, axis=1)
            
            # Metrics
            sq_diff = np.sum((pred_abs - abs_gt)**2, axis=2)
            all_mse.extend(np.mean(sq_diff, axis=1))
            
            c_mse = np.sum((pred_abs[:, -1, :] - abs_gt[:, -1, :])**2, axis=1)
            all_cmse.extend(c_mse)
            
            gt_foot_y = abs_gt[:, -1, 1] + (wh_gt[:, -1, 1] / 2.0)
            pred_foot_y = pred_abs[:, -1, 1] + (wh_gt[:, -1, 1] / 2.0)
            f_mse = (pred_abs[:, -1, 0] - abs_gt[:, -1, 0])**2 + (pred_foot_y - gt_foot_y)**2
            all_cfmse.extend(c_mse + f_mse)

    return np.mean(all_mse), np.mean(all_cmse), np.mean(all_cfmse)

mse, cmse, cfmse = evaluate_pietraj_v2(model, val_loader)
print(f"📊 UPDATED RESULTS:\nMSE: {mse:.2f}\nC-MSE: {cmse:.2f}\nCF-MSE: {cfmse:.2f}")

📊 UPDATED RESULTS:
MSE: 3852.10
C-MSE: 16838.79
CF-MSE: 33677.58
